# COMP3008 Big Data Analytics — Assessment 2
## How has COVID-19 impacted employment patterns and regional economies in the UK?
**Data source:** UK Annual Population Survey (APS), January 2019 – December 2024  
**Module:** COMP3008 | University of Plymouth

---

### How this notebook is organised
| Section | Purpose | Report section it feeds |
|---|---|---|
| 1. Setup | Imports and paths | — |
| 2. Load & Merge | Read both CSVs, combine into one DataFrame | Pre-processing |
| 3. Pre-processing | Column harmonisation, type fixing, missingness | Pre-processing |
| 4. EDA — Method 1 | Descriptive statistics & distributions | Application of Methods |
| 5. EDA — Method 2 | Labour-status time-series trend (2019–2024) | Application of Methods |
| 6. EDA — Method 3 | Regional comparison (employment rate by Government Office Region) | Application of Methods |
| 7. EDA — Method 4 | K-Means clustering of worker profiles | Application of Methods |
| 8. Predictive — ARIMA | Time-series forecast of annual employment rate | Application of Methods |
| 10. Predictive — RF Classifier | Random Forest Classifier — Employment Status Prediction | Application of Methods |
| 11. Model Comparison | Metrics table and suitability commentary | Results |

> **AI declaration:** Sections of this code were drafted with assistance from GitHub Copilot and Claude (A7 — code generation for learning purposes). All code has been reviewed, understood, and adapted by the student.

---
## Section 1 — Setup
Install any missing packages and import everything the notebook needs.  
Run this cell first every time.

In [ ]:
%pip install numpy pandas matplotlib seaborn scikit-learn pmdarima jinja2 --quiet

# ── Standard library ──────────────────────────────────────────────────────────
import warnings
from pathlib import Path

# ── Data ──────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Machine learning ──────────────────────────────────────────────────────────
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ── Time-series ───────────────────────────────────────────────────────────────
# pmdarima is used indirectly via statsmodels ARIMA with manual candidate selection.
# Install with: pip install pmdarima
try:
    import pmdarima  # noqa: F401 — presence check only
    ARIMA_AVAILABLE = True
except ImportError:
    print("pmdarima not found. Run: pip install pmdarima")
    ARIMA_AVAILABLE = False

warnings.filterwarnings("ignore")   # suppress minor sklearn/statsmodels warnings
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 130    # legible at 100% zoom as required by brief

# ── Paths ─────────────────────────────────────────────────────────────────────
# All paths are relative to this notebook's location — no hard-coded drives.
NOTEBOOK_DIR = Path().resolve()                      # wherever the notebook lives
DATA_DIR     = NOTEBOOK_DIR / "Report_Data"          # both CSVs sit here
OUT_DIR      = NOTEBOOK_DIR / "analysis_outputs"     # figures and CSVs written here
FIG_DIR      = OUT_DIR / "figures"                   # all saved figures

OUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

FILE_A = DATA_DIR / "AnnualPopulationSurvey_Jan2019_Dec2021.csv"
FILE_B = DATA_DIR / "AnnualPopulationSurvey_Jan2022_Dec2024.csv"

print("Setup complete.")
print(f"Data directory : {DATA_DIR}")
print(f"Output directory: {OUT_DIR}")

---
## Section 2 — Load & Merge

The APS is split across two files at a period boundary (2019–2021 and 2022–2024).  
Both must be combined into a single DataFrame for longitudinal analysis.  
Column names differ in case and suffix between files (e.g. `GOR9D` vs `GOR9DCENSUS2021`),  
so we normalise everything to **UPPER CASE** before merging.

> **Report link → Pre-processing section:** Describe the merge strategy and the reason for upper-casing here.

In [ ]:
def load_csv(path: Path, period_label: str) -> pd.DataFrame:
    """Read one APS CSV, normalise column names, tag with source period."""
    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.str.upper()   # standardise case across both files
    df["SOURCE_PERIOD"] = period_label    # lets us split back later if needed
    return df

df_a = load_csv(FILE_A, "2019-2021")
df_b = load_csv(FILE_B, "2022-2024")

print(f"File A shape: {df_a.shape}  ({df_a.shape[0]:,} rows, {df_a.shape[1]} columns)")
print(f"File B shape: {df_b.shape}  ({df_b.shape[0]:,} rows, {df_b.shape[1]} columns)")

# Concatenate vertically — columns that don't exist in one file become NaN.
raw = pd.concat([df_a, df_b], ignore_index=True, sort=False)
print(f"\nCombined shape: {raw.shape}  ({raw.shape[0]:,} rows, {raw.shape[1]} columns)")

---
## Section 3 — Pre-processing

### 3a — Column harmonisation

Between 2019–2021 and 2022–2024 the ONS renamed several geographic fields  
to incorporate the 2021 Census geography. We coalesce renamed pairs into  
single canonical columns so downstream code has one consistent name to use.

| Canonical name | 2019–2021 raw | 2022–2024 raw |
|---|---|---|
| `REGION_CODE` | `GOR9D` | `GOR9DCENSUS2021` |
| `COMBINED_AUTHORITY` | `COMBINEDAUTHORITIES` | `COMBINEDAUTHORITIESCENSUS2021` |
| `ITL2_CODE` | `ITL221` | `ITL221CENSUS2021` / `ITL225CENSUS2021` |
| `HIGHEST_QUAL` | `HIQUAL15` | `HIQUAL22` (where HIQUAL15 missing) |

> **Report link → Pre-processing:** Cite the ONS APS technical notes on the 2021 Census geography change.

In [ ]:
def coalesce(df: pd.DataFrame, *cols) -> pd.Series:
    """Return first non-null value across a list of columns (left to right)."""
    existing = [c for c in cols if c in df.columns]
    if not existing:
        return pd.Series(pd.NA, index=df.index)
    result = df[existing[0]].copy()
    for col in existing[1:]:
        result = result.fillna(df[col])
    return result

# ── Build the harmonised core DataFrame ───────────────────────────────────────
df = pd.DataFrame()

df["PERSON_ID"]          = coalesce(raw, "IDREF")
df["SOURCE_PERIOD"]      = raw["SOURCE_PERIOD"]
df["YEAR"]               = pd.to_numeric(coalesce(raw, "FILEYEAR"), errors="coerce")
df["WEIGHT"]             = pd.to_numeric(coalesce(raw, "NPWT22C"),  errors="coerce")
df["AGE"]                = pd.to_numeric(coalesce(raw, "AGE"),      errors="coerce")
df["SEX"]                = coalesce(raw, "SEX").astype("string")
df["LABOUR_STATUS"]      = coalesce(raw, "ILODEFR").astype("string")   # ILO employment status
df["OUTCOME_STATUS"]     = coalesce(raw, "IOUTCOME").astype("string")
df["FULLTIME_PARTTIME"]  = coalesce(raw, "FTPT").astype("string")
df["GROSS_WEEKLY_PAY"]   = pd.to_numeric(coalesce(raw, "GRSSWK"),   errors="coerce")
df["HOURLY_RATE"]        = pd.to_numeric(coalesce(raw, "HRRATE"),   errors="coerce")

# Harmonise a weekly-hours field from common APS hours columns.
df["HOURS_WORKED_WEEK"]  = pd.to_numeric(
    coalesce(raw, "TOTHRS", "USUHR", "ACTHR", "SUMHRS", "TTUSHR"),
    errors="coerce",
)
df.loc[(df["HOURS_WORKED_WEEK"] <= 0) | (df["HOURS_WORKED_WEEK"] > 100), "HOURS_WORKED_WEEK"] = np.nan

# Derived uncapped weekly-pay estimate from hourly rate and hours worked.
df["ESTIMATED_WEEKLY_PAY"] = df["HOURLY_RATE"] * df["HOURS_WORKED_WEEK"]

df["HIGHEST_QUAL"]       = coalesce(raw, "HIQUAL15", "HIQUAL22").astype("string")
df["HEALTH_LIMITATION"]  = coalesce(raw, "HEALYR").astype("string")
df["ETHNICITY"]          = coalesce(raw, "ETH11EW").astype("string")
df["COUNTRY_CODE"]       = coalesce(raw, "CTRY9D").astype("string")
df["COUNTRY_NAME"]       = coalesce(raw, "COUNTRY").astype("string")
df["REGION_CODE"]        = coalesce(raw, "GOR9D", "GOR9DCENSUS2021").astype("string")
df["COMBINED_AUTHORITY"] = coalesce(raw, "COMBINEDAUTHORITIES", "COMBINEDAUTHORITIESCENSUS2021").astype("string")
df["ITL2_CODE"]          = coalesce(raw, "ITL221", "ITL221CENSUS2021", "ITL225CENSUS2021").astype("string")
df["ITL3_CODE"]          = coalesce(raw, "ITL321", "ITL321CENSUS2021", "ITL325CENSUS2021").astype("string")

# Prepare time-series inputs here so the APS files are only loaded once.
ts_raw = raw[["FILEYEAR", "REFDTE", "ILODEFR"]].copy()
ts_raw["EMPLOYED"] = ts_raw["ILODEFR"].astype("string").isin(["1", "4"]).astype(int)
refdte_str = ts_raw["REFDTE"].astype("string").str.replace(r"\.0$", "", regex=True).str.zfill(8)
ts_raw["REF_DATE"] = pd.to_datetime(refdte_str, format="%d%m%Y", errors="coerce")
ts_raw = ts_raw.dropna(subset=["REF_DATE"]).copy()
ts_raw["MONTH"] = ts_raw["REF_DATE"].dt.to_period("M").dt.to_timestamp()
monthly_emp = ts_raw.groupby("MONTH")["EMPLOYED"].mean().mul(100).sort_index()

annual_work = df.dropna(subset=["YEAR", "LABOUR_STATUS"]).copy()
annual_work["YEAR"] = pd.to_numeric(annual_work["YEAR"], errors="coerce").astype("Int64")
annual_work = annual_work.dropna(subset=["YEAR"])
annual_work["EMPLOYED"] = annual_work["LABOUR_STATUS"].astype("string").isin(["1", "4"]).astype(int)
annual_emp = annual_work.groupby("YEAR")["EMPLOYED"].mean().mul(100).sort_index()
annual_emp.index = annual_emp.index.astype(int)

del raw, df_a, df_b   # free memory — we only need the harmonised df from here

print(f"Harmonised DataFrame: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("Coverage checks:")
print(f"  HOURLY_RATE non-missing       : {df['HOURLY_RATE'].notna().sum():,}")
print(f"  HOURS_WORKED_WEEK non-missing : {df['HOURS_WORKED_WEEK'].notna().sum():,}")
print(f"  ESTIMATED_WEEKLY_PAY non-missing: {df['ESTIMATED_WEEKLY_PAY'].notna().sum():,}")
df.dtypes
print("Checks:")
print(df["REGION_CODE"].value_counts().head(20))
print(df["LABOUR_STATUS"].value_counts().head(10))
print(f"Prepared monthly series: {len(monthly_emp)} months ({monthly_emp.index.min().date()} to {monthly_emp.index.max().date()})")
print(f"Prepared annual series : {len(annual_emp)} years ({annual_emp.index.min()} to {annual_emp.index.max()})")
print("end")

### 3b — Missingness audit

Before any analysis, we must understand which variables have usable coverage.  
Variables with very high missingness are retained but will be excluded from  
models where they would cause problems.

> **Report link → Pre-processing:** Use the table below to justify any variables excluded from modeling.

In [ ]:
miss = (df.isna().mean() * 100).sort_values(ascending=False).round(2)
miss_df = miss.reset_index()
miss_df.columns = ["Variable", "Missing %"]

# Colour-code: red > 70%, amber 30–70%, green < 30%
def colour_missing(val):
    if val > 70:   return "background-color: #f4cccc"
    if val > 30:   return "background-color: #fce8b2"
    return "background-color: #d9ead3"

miss_df.style.map(colour_missing, subset=["Missing %"])

### 3c — Data quality summary
Quick check on row counts, year coverage, and duplicate records.

In [ ]:
print("=== Data Quality Summary ===")
print(f"Total rows            : {len(df):,}")
print(f"Duplicate rows        : {df.duplicated().sum():,}")
print(f"Year range            : {int(df['YEAR'].min())} – {int(df['YEAR'].max())}")
print(f"Rows per year:")
print(df["YEAR"].value_counts().sort_index().to_string())
print(f"\nUnique REGION_CODE values : {df['REGION_CODE'].nunique()}")
print(f"Unique COUNTRY_NAME values: {df['COUNTRY_NAME'].unique()}")

df.to_csv(OUT_DIR / "harmonised_aps_data.csv", index=False)
print(f"Saved → {OUT_DIR / 'harmonised_aps_data.csv'}")

In [ ]:
# Time-series inputs are prepared in the harmonisation cell above so the APS files are only loaded once.
if "monthly_emp" not in globals() or "annual_emp" not in globals():
    raise RuntimeError("Run the harmonisation cell first: monthly_emp and annual_emp are prepared there.")

print(f"Prepared monthly series: {len(monthly_emp)} months ({monthly_emp.index.min().date()} to {monthly_emp.index.max().date()})")
print(f"Prepared annual series : {len(annual_emp)} years ({annual_emp.index.min()} to {annual_emp.index.max()})")

### 3d — Time-series inputs for forecasting

Monthly and annual employment-rate series are prepared in the harmonisation cell above,  
so the forecasting section only fits models and plots results (no repeated data wrangling).

---
## Section 4 — EDA Method 1: Descriptive Statistics & Distributions

**Why this method?**  
Descriptive statistics provide the baseline understanding of each variable's  
central tendency, spread, and shape before any modelling begins.  
Age and gross weekly pay distributions are particularly relevant to the  
research question — COVID affected different age groups and pay bands unevenly.

> **Report link → Application of Methods:** Discuss what the age distribution tells us  
> about the working-age population in the sample.

In [ ]:
# ── Summary statistics for key numeric variables ───────────────────────────────
numeric_vars = [
    "AGE",
    "GROSS_WEEKLY_PAY",
    "HOURLY_RATE",
    "HOURS_WORKED_WEEK",
    "ESTIMATED_WEEKLY_PAY",
    "WEIGHT",
]
df[numeric_vars].describe().round(2)

In [ ]:
# ── Diagnostics: effective sample size and pay-shape checks ─────────────────────
diag_vars = [
    "AGE",
    "GROSS_WEEKLY_PAY",
    "HOURLY_RATE",
    "HOURS_WORKED_WEEK",
    "ESTIMATED_WEEKLY_PAY",
    "WEIGHT",
]
diag = pd.DataFrame({
    "non_missing_n": df[diag_vars].notna().sum(),
    "missing_pct": (df[diag_vars].isna().mean() * 100).round(2),
})
diag["coverage_pct"] = (100 - diag["missing_pct"]).round(2)
diag = diag.sort_values("missing_pct", ascending=False)

print("Section 4 diagnostics (coverage and missingness):")
display(diag)

pay_obs = pd.to_numeric(df["GROSS_WEEKLY_PAY"], errors="coerce").dropna()
pay_est = pd.to_numeric(df["ESTIMATED_WEEKLY_PAY"], errors="coerce").dropna()

print("\nObserved gross weekly pay diagnostics:")
print(f"Non-missing rows: {len(pay_obs):,} / {len(df):,} ({100*len(pay_obs)/len(df):.2f}% coverage)")
print(f"95th percentile: {pay_obs.quantile(0.95):.2f}")
print(f"99th percentile: {pay_obs.quantile(0.99):.2f}")
print(f"Maximum value: {pay_obs.max():.2f}")
print("Top 10 most frequent values:")
print(pay_obs.value_counts().head(10).to_string())

if np.isclose(pay_obs.quantile(0.99), pay_obs.max()):
    print("\nNote: p99 equals the maximum observed value, so 99th-percentile trimming removes little or nothing.")
    print("This suggests top-coding/heaping at the cap, not only rare extreme outliers.")

print("\nDerived estimated weekly pay diagnostics (HOURLY_RATE × HOURS_WORKED_WEEK):")
print(f"Non-missing rows: {len(pay_est):,} / {len(df):,} ({100*len(pay_est)/len(df):.2f}% coverage)")
print(f"Mean: {pay_est.mean():.2f}")
print(f"Median: {pay_est.median():.2f}")
print(f"95th percentile: {pay_est.quantile(0.95):.2f}")
print(f"99th percentile: {pay_est.quantile(0.99):.2f}")
print(f"Maximum value: {pay_est.max():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Age distribution ──────────────────────────────────────────────────────────
age = df["AGE"].dropna()
sns.histplot(age, bins=40, kde=True, ax=axes[0], color="#4878CF")
axes[0].set_title("Age Distribution of APS Respondents (2019–2024)")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Count")
axes[0].axvline(age.mean(), color="red", linestyle="--", label=f"Mean: {age.mean():.1f}")
axes[0].legend()

# ── Gross weekly pay distribution (trimmed at 99th percentile to remove outliers)
pay = df["GROSS_WEEKLY_PAY"].dropna()
pay_trimmed = pay[pay <= pay.quantile(0.99)]
sns.histplot(pay_trimmed, bins=50, kde=True, ax=axes[1], color="#6ACC65")
axes[1].set_title("Gross Weekly Pay Distribution (trimmed at 99th pct)")
axes[1].set_xlabel("Gross Weekly Pay (£)")
axes[1].set_ylabel("Count")
axes[1].axvline(pay_trimmed.median(), color="red", linestyle="--",
                label=f"Median: £{pay_trimmed.median():.0f}")
axes[1].legend()

plt.tight_layout()
fig.savefig(FIG_DIR / "eda1_distributions.png", dpi=150)
plt.show()
print(f"Saved → {FIG_DIR / 'eda1_distributions.png'}")

---
## Section 5 — EDA Method 2: Labour Status Time-Series (2019–2024)

**Why this method?**  
A grouped time-series plot shows *how the composition of employment changed  
year-by-year*, making the COVID shock (2020) and recovery (2021–2022) directly  
visible. `ILODEFR` (mapped here as `LABOUR_STATUS`) is the standard ILO  
definition of employment status used by the ONS.

**ILODEFR code meanings (from data dictionary):**  
1 = In employment, 2 = ILO unemployed, 3 = Economically inactive

> **Report link → Application of Methods:** Identify the 2020 dip in employment and  
> discuss how furlough may have affected the ILO classification.

In [ ]:
# Recode numeric codes to readable labels for the chart
STATUS_LABELS = {
    "1": "In Employment",
    "2": "ILO Unemployed",
    "3": "Economically Inactive",
    "4": "Government-Supported Training",
}

ts = df.dropna(subset=["YEAR", "LABOUR_STATUS"]).copy()
ts["YEAR"] = ts["YEAR"].astype(int)
ts["STATUS_LABEL"] = ts["LABOUR_STATUS"].map(STATUS_LABELS).fillna("Other")

# Calculate percentage share of each status per year
grouped = (
    ts.groupby(["YEAR", "STATUS_LABEL"])
    .size()
    .reset_index(name="count")
)
totals = grouped.groupby("YEAR")["count"].transform("sum")
grouped["pct"] = 100 * grouped["count"] / totals

pivot = grouped.pivot(index="YEAR", columns="STATUS_LABEL", values="pct").fillna(0)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
pivot.plot(ax=ax, marker="o", linewidth=2)
ax.axvspan(2019.5, 2021.5, alpha=0.08, color="red", label="COVID period (2020–2021)")
ax.set_title("Labour Status Composition by Year (2019–2024)")
ax.set_xlabel("Year")
ax.set_ylabel("Share of Respondents (%)")
ax.set_xticks(sorted(ts["YEAR"].unique()))
ax.legend(title="Labour Status", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
fig.savefig(FIG_DIR / "eda2_labour_status_timeseries.png", dpi=150)
plt.show()
print(f"Saved → {FIG_DIR / 'eda2_labour_status_timeseries.png'}")

---
## Section 6 — EDA Method 3: Regional Comparison

**Why this method?**  
COVID's economic impact was not uniform across UK regions.  
A regional comparison reveals which Government Office Regions (GORs)  
experienced the greatest employment rate drops and slowest recoveries,  
directly addressing the *regional economies* component of the research question.

> **Report link → Application of Methods:** Compare London vs. the North to discuss  
> structural differences in regional labour markets.

In [ ]:
# GOR9D region codes (England + Wales + Scotland + NI)
# Note: Scotland, Wales, and NI use aggregate "99999999" codes in GOR9D, not the census boundary codes.
REGION_NAMES = {
    "E12000001": "North East",
    "E12000002": "North West",
    "E12000003": "Yorkshire & Humber",
    "E12000004": "East Midlands",
    "E12000005": "West Midlands",
    "E12000006": "East of England",
    "E12000007": "London",
    "E12000008": "South East",
    "E12000009": "South West",
    "W99999999": "Wales",
    "S99999999": "Scotland",
    "N99999999": "Northern Ireland",
}

reg = df.dropna(subset=["YEAR", "REGION_CODE", "LABOUR_STATUS"]).copy()
reg["YEAR"] = reg["YEAR"].astype(int)
reg["REGION_NAME"] = reg["REGION_CODE"].map(REGION_NAMES)

# Employment rate = proportion of respondents coded as 'In Employment' (ILODEFR = 1)
reg["EMPLOYED"] = reg["LABOUR_STATUS"].isin(["1", "4"]).astype(int)

emp_rate = (
    reg.groupby(["YEAR", "REGION_NAME"])["EMPLOYED"]
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={"EMPLOYED": "Employment Rate (%)"})
)

# ── Heatmap: regions × years ───────────────────────────────────────────────────
pivot_reg = emp_rate.pivot(index="REGION_NAME", columns="YEAR", values="Employment Rate (%)")

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(
    pivot_reg,
    annot=True, fmt=".1f",
    cmap="RdYlGn",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Employment Rate (%)"},
)
ax.set_title("Employment Rate (%) by Region and Year (All UK Nations)")
ax.set_xlabel("Year")
ax.set_ylabel("Region")
plt.tight_layout()
fig.savefig(FIG_DIR / "eda3_regional_employment_heatmap.png", dpi=150)
plt.show()
print(f"Saved → {FIG_DIR / 'eda3_regional_employment_heatmap.png'}")

---
## Section 7 — EDA Method 4: K-Means Clustering of Worker Profiles

**Why this method?**  
K-Means clustering is an unsupervised method that groups respondents by  
numerical similarity (age, pay, hours). It reveals distinct worker *profiles*  
without assuming categories in advance — useful for checking whether  
COVID affected some clusters more than others.  
PCA is used only to visualise the 5-dimensional cluster space in 2D.

> **Report link → Application of Methods:** Describe each cluster by its centroid  
> values. Discuss whether cluster membership shifted between 2019 and 2021.

In [ ]:
# Features for clustering — only numeric, employment-relevant variables
CLUSTER_FEATURES = ["AGE", "HOURS_WORKED_WEEK", "GROSS_WEEKLY_PAY", "HOURLY_RATE"]

cluster_work = df[CLUSTER_FEATURES].apply(pd.to_numeric, errors="coerce")

# Use a random sample for speed — 50,000 rows is representative of 750k
sample_n = min(50_000, len(cluster_work))
cluster_sample = cluster_work.sample(n=sample_n, random_state=42)

# ── Elbow method to choose k ───────────────────────────────────────────────────
imputer = SimpleImputer(strategy="median")   # fill missing values with median
scaler  = StandardScaler()                   # standardise so large-scale pay doesn't dominate

X = scaler.fit_transform(imputer.fit_transform(cluster_sample))

inertias = []
K_RANGE  = range(2, 9)
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(K_RANGE), inertias, marker="o")
ax.set_title("Elbow Plot — Choosing Number of Clusters (k)")
ax.set_xlabel("k (number of clusters)")
ax.set_ylabel("Inertia")
plt.tight_layout()
fig.savefig(FIG_DIR / "eda4a_elbow.png", dpi=150)
plt.show()
print("Choose k at the 'elbow' — the point where inertia stops decreasing steeply.")

In [ ]:
# ── Fit final KMeans with chosen k ────────────────────────────────────────────
# Change K_CHOSEN here if your elbow plot suggests a different value.
K_CHOSEN = 4

kmeans = KMeans(n_clusters=K_CHOSEN, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)

# ── Reduce to 2D with PCA for visualisation only ───────────────────────────────
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X)

fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap="Set2", s=8, alpha=0.5)
ax.set_title(f"K-Means Clusters (k={K_CHOSEN}) — Visualised in PCA Space")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
legend_handles, _ = scatter.legend_elements()
ax.legend(legend_handles, [f"Cluster {i}" for i in range(K_CHOSEN)],
          title="Cluster", loc="best")
plt.tight_layout()
fig.savefig(FIG_DIR / "eda4b_kmeans_pca.png", dpi=150)
plt.show()

# ── Cluster profile summary ───────────────────────────────────────────────────
# Shows the mean of each feature per cluster — helps interpret what each group represents.
cluster_sample_with_labels = cluster_sample.copy()
cluster_sample_with_labels["Cluster"] = labels
print("\nCluster centroids (original scale):")
cluster_sample_with_labels.groupby("Cluster").mean().round(2)

---
## Section 8 — Predictive Method 1: ARIMA Time-Series Forecast

**Why ARIMA?**  
ARIMA models temporal dependence in a univariate sequence. We use the **monthly**  
employment-rate series prepared in the harmonisation cell in Section 3 from APS data.

The workflow below:
- validates on 2024 monthly values (trained on 2019–2023)
- compares a small set of **ARIMA-only** (non-seasonal) specifications
- prefers models with good validation error and non-degenerate future shape
- forecasts the next **24 months** (2025–2026) with confidence intervals

**Limitation to discuss in report:**  
This remains a relatively short macro series and may not capture structural breaks.  
Forecasts are indicative rather than causal.

> **Report link → Application of Methods + Results:** Report the selected ARIMA  
> order/trend, 2024 validation MAE, and interpret the 2025–2026 projected path.

In [ ]:
if not ARIMA_AVAILABLE:
    print("Skipping ARIMA — install pmdarima first: pip install pmdarima")
else:
    from statsmodels.tsa.arima.model import ARIMA

    # Use pre-processed time-series inputs from the harmonisation cell in Section 3.
    if "monthly_emp" not in globals():
        raise RuntimeError("Run the harmonisation cell in Section 3 first: monthly_emp is not prepared.")

    print(f"Monthly employment series length: {len(monthly_emp)} months")
    print(monthly_emp.tail(6).round(2).to_string())

    # Validation split: train through 2023, validate on 2024.
    train_ts = monthly_emp[monthly_emp.index <= "2023-12-31"]
    test_ts = monthly_emp[
        (monthly_emp.index >= "2024-01-01") & (monthly_emp.index <= "2024-12-31")
    ]

    # Future horizon: next 24 months beyond latest observed month.
    FUTURE_HORIZON = 24
    last_month = monthly_emp.index.max()
    future_months = pd.date_range(last_month + pd.offsets.MonthBegin(1), periods=FUTURE_HORIZON, freq="MS")

    # ARIMA-only candidate set (non-seasonal).
    candidates = [
        {"order": (1, 0, 0), "trend": "c"},
        {"order": (2, 0, 0), "trend": "c"},
        {"order": (1, 1, 0), "trend": "t"},
        {"order": (1, 1, 1), "trend": "t"},
        {"order": (2, 1, 0), "trend": "t"},
        {"order": (2, 1, 1), "trend": "t"},
        {"order": (3, 1, 0), "trend": "t"},
        {"order": (0, 1, 1), "trend": "t"},
    ]

    leaderboard = []
    flat_threshold = 0.02  # std threshold for non-degenerate 24-month forecast

    for spec in candidates:
        try:
            model = ARIMA(train_ts, order=spec["order"], trend=spec["trend"])
            res = model.fit()
            val_pred = pd.Series(res.get_forecast(steps=len(test_ts)).predicted_mean, index=test_ts.index)
            eval_df = pd.concat([test_ts.rename("actual"), val_pred.rename("pred")], axis=1).dropna()
            if len(eval_df) == 0:
                continue
            mae = mean_absolute_error(eval_df["actual"], eval_df["pred"])

            # Fit same spec on full data and assess future profile flatness.
            full_model_tmp = ARIMA(monthly_emp, order=spec["order"], trend=spec["trend"])
            full_res_tmp = full_model_tmp.fit()
            fut_tmp = pd.Series(full_res_tmp.get_forecast(steps=FUTURE_HORIZON).predicted_mean, index=future_months)
            fut_std = float(pd.to_numeric(fut_tmp, errors="coerce").std())

            leaderboard.append({
                "order": spec["order"],
                "trend": spec["trend"],
                "mae": float(mae),
                "future_std": fut_std,
            })
        except Exception:
            continue

    if len(leaderboard) == 0:
        raise RuntimeError("No ARIMA candidate could be fitted successfully.")

    lb = pd.DataFrame(leaderboard).sort_values(["mae", "future_std"], ascending=[True, False]).reset_index(drop=True)
    print("\nARIMA candidate leaderboard (top 5):")
    print(lb.head(5).to_string(index=False))

    non_flat = lb[lb["future_std"] >= flat_threshold]
    if len(non_flat) > 0:
        chosen = non_flat.iloc[0]
        print("\nChoosing best non-flat ARIMA model by validation MAE.")
    else:
        chosen = lb.iloc[0]
        print("\nNo candidate produced strong non-flat dynamics; choosing best MAE ARIMA model.")

    best_order = tuple(chosen["order"])
    best_trend = str(chosen["trend"])

    print(f"Selected ARIMA model: order={best_order}, trend='{best_trend}'")

    # Final validation fit on train period.
    best_val_model = ARIMA(train_ts, order=best_order, trend=best_trend)
    best_val_res = best_val_model.fit()
    test_forecast = pd.Series(best_val_res.get_forecast(steps=len(test_ts)).predicted_mean, index=test_ts.index)
    test_forecast = pd.to_numeric(test_forecast, errors="coerce")
    if test_forecast.isna().any():
        test_forecast = test_forecast.fillna(float(train_ts.dropna().iloc[-1]))

    eval_df = pd.concat([test_ts.rename("actual"), test_forecast.rename("pred")], axis=1).dropna()
    if len(eval_df) == 0:
        arima_mae = float("nan")
        print("Monthly ARIMA MAE on 2024 validation: unavailable (no valid overlap after cleaning)")
    else:
        arima_mae = mean_absolute_error(eval_df["actual"], eval_df["pred"])
        print(f"Monthly ARIMA MAE on 2024 validation: {arima_mae:.3f} percentage points")

    # Refit selected ARIMA model on full monthly data for future forecast.
    best_full_model = ARIMA(monthly_emp, order=best_order, trend=best_trend)
    best_full_res = best_full_model.fit()
    future_obj = best_full_res.get_forecast(steps=FUTURE_HORIZON)
    future_forecast = pd.Series(future_obj.predicted_mean, index=future_months)
    future_ci = future_obj.conf_int()
    future_lower = pd.Series(future_ci.iloc[:, 0].values, index=future_months)
    future_upper = pd.Series(future_ci.iloc[:, 1].values, index=future_months)

    future_forecast = pd.to_numeric(future_forecast, errors="coerce")
    future_lower = pd.to_numeric(future_lower, errors="coerce").fillna(future_forecast)
    future_upper = pd.to_numeric(future_upper, errors="coerce").fillna(future_forecast)
    if future_forecast.isna().any():
        future_forecast = future_forecast.fillna(float(monthly_emp.dropna().iloc[-1]))

    print("\nFuture monthly ARIMA forecast (first 12 months):")
    print(future_forecast.head(12).round(2).to_string())

    # Plot actual + validation forecast + future forecast.
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(monthly_emp.index, monthly_emp.values, label="Actual (Monthly)", color="#4878CF", linewidth=1.8)
    ax.plot(
        test_ts.index,
        test_forecast.values,
        linestyle="--",
        label="ARIMA Validation Forecast (2024)",
        color="#E06C75",
        linewidth=2,
    )
    ax.plot(
        future_months,
        future_forecast.values,
        linestyle="--",
        label="ARIMA Future Forecast (Next 24 Months)",
        color="#55A868",
        linewidth=2,
    )
    ax.fill_between(
        future_months,
        future_lower.values,
        future_upper.values,
        color="#55A868",
        alpha=0.15,
        label="Future forecast interval",
    )

    ax.axvspan(pd.Timestamp("2024-01-01"), pd.Timestamp("2024-12-31"), alpha=0.07, color="grey", label="Validation window")
    ax.axvspan(future_months.min(), future_months.max(), alpha=0.07, color="#55A868", label="Future window")
    ax.set_title("ARIMA Forecast of UK Employment Rate (Monthly)")
    ax.set_xlabel("Month")
    ax.set_ylabel("Employment Rate (%)")
    ax.legend(loc="best")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "model1_arima_forecast.png", dpi=150)
    plt.show()
    print(f"Saved → {FIG_DIR / 'model1_arima_forecast.png'}")

---
## Section 10 — Predictive Method 2: Random Forest Classifier (Employment Status)

**Why Random Forest Classifier?**  
Rather than predicting a continuous wage outcome — which suffers from 72% missingness in APS pay data — a classification approach predicts *employment status*, a complete, policy-relevant outcome available for all 750,936 respondents. This directly addresses whether COVID-19 affected an individual's probability of being in employment.

**Target variable:**  
Binary employment status derived from `LABOUR_STATUS` (ILO definition):  
- 1 = Employed (codes "1" = in employment, "4" = government-supported training)  
- 0 = Not employed (codes "2" = ILO unemployed, "3" = economically inactive)

**Train/test split:**  
Train on 2019–2022 (pre/during/early-recovery period), test on 2023–2024 (full post-COVID period). This mimics a real-world scenario where the model trained on earlier data must generalise to the most recent years.

> **Report link → Results:** Discuss accuracy and F1 score, what the feature importances reveal about predictors of employment, and what the confusion matrix tells us about misclassification patterns.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
from collections import defaultdict

# ── Pipeline helper ───────────────────────────────────────────────────────────
def build_clf_pipeline(model, numeric_features, cat_features):
    num_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale",  StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore")),
    ])
    pre = ColumnTransformer([
        ("num", num_pipe, numeric_features),
        ("cat", cat_pipe, cat_features),
    ])
    return Pipeline([("pre", pre), ("model", model)])

# ── Target: binary employment status ─────────────────────────────────────────
# Employed = LABOUR_STATUS "1" (in employment) or "4" (govt-supported training)
# Not employed = "2" (ILO unemployed) or "3" (economically inactive)
clf_df = df.copy()
clf_df["EMPLOYED"] = clf_df["LABOUR_STATUS"].isin(["1", "4"]).astype(int)

CLF_NUM_FEATURES = ["AGE", "YEAR"]
CLF_CAT_FEATURES = ["SEX", "HIGHEST_QUAL", "ETHNICITY", "REGION_CODE", "COUNTRY_CODE"]

clf_df = clf_df[CLF_NUM_FEATURES + CLF_CAT_FEATURES + ["EMPLOYED"]].dropna(subset=["AGE", "YEAR"])

print(f"Total rows for classifier: {len(clf_df):,}")
vc = clf_df["EMPLOYED"].value_counts()
print(f"\nClass balance:")
print(f"  Not Employed (0): {vc.get(0, 0):,}  ({100 * vc.get(0, 0) / len(clf_df):.1f}%)")
print(f"  Employed     (1): {vc.get(1, 0):,}  ({100 * vc.get(1, 0) / len(clf_df):.1f}%)")

# Time-aware split: train on 2019–2022, test on 2023–2024
clf_train = clf_df[clf_df["YEAR"] <= 2022]
clf_test  = clf_df[clf_df["YEAR"].isin([2023, 2024])]
print(f"\nTrain rows (≤ 2022): {len(clf_train):,} | Test rows (2023–2024): {len(clf_test):,}")

X_clf_train = clf_train[CLF_NUM_FEATURES + CLF_CAT_FEATURES].copy()
y_clf_train = clf_train["EMPLOYED"]
X_clf_test  = clf_test[CLF_NUM_FEATURES + CLF_CAT_FEATURES].copy()
y_clf_test  = clf_test["EMPLOYED"]

for col in CLF_CAT_FEATURES:
    X_clf_train[col] = X_clf_train[col].astype("object").replace({pd.NA: np.nan})
    X_clf_test[col]  = X_clf_test[col].astype("object").replace({pd.NA: np.nan})

# Cap training at 200,000 rows
MAX_TRAIN_CLF = 200_000
if len(X_clf_train) > MAX_TRAIN_CLF:
    sub_idx = X_clf_train.sample(n=MAX_TRAIN_CLF, random_state=42).index
    X_clf_train = X_clf_train.loc[sub_idx]
    y_clf_train = y_clf_train.loc[sub_idx]
    print(f"Training capped at {MAX_TRAIN_CLF:,} rows (random_state=42)")

clf_pipe = build_clf_pipeline(
    RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    ),
    CLF_NUM_FEATURES,
    CLF_CAT_FEATURES,
)
clf_pipe.fit(X_clf_train, y_clf_train)
pred_clf = clf_pipe.predict(X_clf_test)

clf_accuracy = accuracy_score(y_clf_test, pred_clf)
clf_f1       = f1_score(y_clf_test, pred_clf)

print(f"\nClassifier Accuracy : {clf_accuracy:.4f}")
print(f"Classifier F1 Score : {clf_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_clf_test, pred_clf, target_names=["Not Employed", "Employed"]))

In [ ]:
# ── Feature importance — aggregated by original feature, human-readable labels ─
pre_step  = clf_pipe.named_steps["pre"]
cat_names = pre_step.named_transformers_["cat"]["encode"].get_feature_names_out(CLF_CAT_FEATURES)
all_feature_names = CLF_NUM_FEATURES + list(cat_names)
importances = clf_pipe.named_steps["model"].feature_importances_

# Sum importances across all OHE dummies back to the original feature name
imp_by_feature = defaultdict(float)
for fname, imp in zip(all_feature_names, importances):
    group = next((f for f in CLF_NUM_FEATURES + CLF_CAT_FEATURES if fname == f or fname.startswith(f + "_")), fname)
    imp_by_feature[group] += imp

FEATURE_LABELS = {
    "AGE":          "Age",
    "YEAR":         "Year",
    "SEX":          "Sex",
    "HIGHEST_QUAL": "Highest Qualification",
    "ETHNICITY":    "Ethnicity",
    "REGION_CODE":  "Region",
    "COUNTRY_CODE": "Country",
}

imp_df = (
    pd.DataFrame([{"Feature": FEATURE_LABELS.get(k, k), "Importance": v} for k, v in imp_by_feature.items()])
    .sort_values("Importance", ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=imp_df, x="Importance", y="Feature", ax=ax, palette="viridis")
ax.set_title("Random Forest Classifier: Feature Importances")
ax.set_xlabel("Mean Decrease in Impurity (summed across categories)")
ax.set_ylabel("")
plt.tight_layout()
fig.savefig(FIG_DIR / "model2_rf_employment_classifier_importance.png", dpi=150)
plt.show()
print(f"Saved → {FIG_DIR} / 'model2_rf_employment_classifier_importance.png'")

In [ ]:
# ── Diagnostic: raw individual feature importances before aggregation ─────────
raw_imp_df = (
    pd.DataFrame({"Feature": all_feature_names, "Importance": importances})
    .sort_values("Importance", ascending=False)
    .head(20)
)
print("Top 20 individual feature importances (pre-aggregation):")
print(raw_imp_df.to_string(index=False))

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
cm   = confusion_matrix(y_clf_test, pred_clf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not Employed", "Employed"])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Random Forest Classifier: Confusion Matrix (Test Set 2023–2024)")
plt.tight_layout()
fig.savefig(FIG_DIR / "model2_rf_employment_classifier_confusion.png", dpi=150)
plt.show()
print(f"Saved → {FIG_DIR} / 'model2_rf_employment_classifier_confusion.png'")

In [ ]:
# ── ROC Curve ─────────────────────────────────────────────────────────────────
from sklearn.metrics import roc_curve, auc

prob_clf = clf_pipe.predict_proba(X_clf_test)[:, 1]
fpr, tpr, _ = roc_curve(y_clf_test, prob_clf)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color="#4878CF", linewidth=2, label=f"ROC curve (AUC = {roc_auc:.3f})")
ax.plot([0, 1], [0, 1], "r--", linewidth=1, label="Random classifier")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — Random Forest Employment Classifier (Test Set 2023–2024)")
ax.legend()
plt.tight_layout()
fig.savefig(FIG_DIR / "model2_rf_roc_curve.png", dpi=150)
plt.show()
print(f"AUC: {roc_auc:.4f}")
print(f"Saved → {FIG_DIR} / 'model2_rf_roc_curve.png'")

In [ ]:
# ── Actual vs Predicted employment rate by year (all years, full dataset) ─────
X_all = clf_df[CLF_NUM_FEATURES + CLF_CAT_FEATURES].copy()
for col in CLF_CAT_FEATURES:
    X_all[col] = X_all[col].astype("object").replace({pd.NA: np.nan})

prob_all  = clf_pipe.predict_proba(X_all)[:, 1]
pred_all  = clf_pipe.predict(X_all)

year_comp = clf_df[["YEAR", "EMPLOYED"]].copy()
year_comp["Predicted"] = prob_all
year_comp["YEAR"] = year_comp["YEAR"].astype(int)

year_summary = (
    year_comp.groupby("YEAR")
    .agg(Actual=("EMPLOYED", "mean"), Predicted=("Predicted", "mean"))
    .mul(100)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(year_summary["YEAR"], year_summary["Actual"],    marker="o", linewidth=2,
        label="Actual Employment Rate",    color="#4878CF")
ax.plot(year_summary["YEAR"], year_summary["Predicted"], marker="s", linewidth=2,
        linestyle="--", label="Predicted Employment Rate", color="#E06C75")
ax.axvspan(2019.5, 2021.5, alpha=0.08, color="red", label="COVID period (2020–2021)")
ax.axvline(2022.5, color="grey", linestyle=":", linewidth=1.5, label="Train / test boundary")
ax.set_title("Actual vs Predicted Employment Rate by Year")
ax.set_xlabel("Year")
ax.set_ylabel("Employment Rate (%)")
ax.set_xticks(year_summary["YEAR"])
ax.legend()
plt.tight_layout()
fig.savefig(FIG_DIR / "model2_rf_employment_rate_by_year.png", dpi=150)
plt.show()
print(f"Saved → {FIG_DIR} / 'model2_rf_employment_rate_by_year.png'")

In [ ]:
# ── Misclassification rate by year ────────────────────────────────────────────
year_err = clf_df[["YEAR", "EMPLOYED"]].copy()
year_err["Correct"] = (pred_all == clf_df["EMPLOYED"].values).astype(int)
year_err["YEAR"] = year_err["YEAR"].astype(int)

err_by_year = year_err.groupby("YEAR")["Correct"].mean().reset_index()
err_by_year["Error Rate (%)"] = (1 - err_by_year["Correct"]) * 100

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(err_by_year["YEAR"], err_by_year["Error Rate (%)"], color="#4878CF", alpha=0.8)
ax.axvspan(2019.5, 2021.5, alpha=0.08, color="red", label="COVID period (2020–2021)")
ax.axvline(2022.5, color="grey", linestyle=":", linewidth=1.5, label="Train / test boundary")
ax.set_title("Misclassification Rate by Year")
ax.set_xlabel("Year")
ax.set_ylabel("Error Rate (%)")
ax.set_xticks(err_by_year["YEAR"])
ax.legend()
plt.tight_layout()
fig.savefig(FIG_DIR / "model2_rf_error_by_year.png", dpi=150)
plt.show()
print(f"Saved → {FIG_DIR} / 'model2_rf_error_by_year.png'")

In [ ]:
# ── Actual vs predicted employment rate by age group ─────────────────────────
age_comp = clf_df[["AGE", "EMPLOYED"]].copy()
age_comp["Predicted"] = prob_all
age_comp["Age Group"] = pd.cut(
    age_comp["AGE"],
    bins=[0, 15, 24, 34, 44, 54, 64, 100],
    labels=["0–15", "16–24", "25–34", "35–44", "45–54", "55–64", "65+"],
)

age_summary = (
    age_comp.groupby("Age Group", observed=True)
    .agg(Actual=("EMPLOYED", "mean"), Predicted=("Predicted", "mean"))
    .mul(100)
    .reset_index()
)

x = range(len(age_summary))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i - width / 2 for i in x], age_summary["Actual"],    width, label="Actual",    color="#4878CF", alpha=0.85)
ax.bar([i + width / 2 for i in x], age_summary["Predicted"], width, label="Predicted", color="#E06C75", alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels(age_summary["Age Group"])
ax.set_title("Actual vs Predicted Employment Rate by Age Group")
ax.set_xlabel("Age Group")
ax.set_ylabel("Employment Rate (%)")
ax.legend()
plt.tight_layout()
fig.savefig(FIG_DIR / "model2_rf_employment_by_age.png", dpi=150)
plt.show()
print(f"Saved → {FIG_DIR} / 'model2_rf_employment_by_age.png'")

---
## Section 11 — Model Comparison & Suitability

This section produces the summary table required by the brief's instruction to  
*'comment on suitability and accuracy of each predictive method'*.

> **Report link → Results:** Compare ARIMA and the Random Forest Classifier on their respective tasks. Discuss what each model reveals about COVID-19's impact — ARIMA on the aggregate employment rate trend, the classifier on individual-level employment probability.

In [ ]:
rows = [
    {
        "Model":       "ARIMA",
        "Target":      "Monthly employment rate (%)",
        "Accuracy":    "N/A",
        "F1 Score":    "N/A",
        "MAE":         f"{arima_mae:.3f} pp" if (ARIMA_AVAILABLE and 'arima_mae' in dir()) else "N/A",
        "Suitability": "Appropriate for monthly trend dynamics; limited by short macro history and structural-break risk.",
    },
    {
        "Model":       "Random Forest Classifier",
        "Target":      "Binary employment status (Employed vs Not Employed)",
        "Accuracy":    f"{clf_accuracy:.4f}",
        "F1 Score":    f"{clf_f1:.4f}",
        "MAE":         "N/A",
        "Suitability": "Captures non-linear patterns across the full 750k APS sample; no pay-data filtering required.",
    },
]

comparison_df = pd.DataFrame(rows).set_index("Model")
comparison_df.to_csv(OUT_DIR / "model_comparison.csv")
print("Model Comparison Table:")
comparison_df

---
## End of Notebook

All figures are saved to `analysis_outputs/figures/`.  
The model comparison table is saved to `analysis_outputs/model_comparison.csv`.

### Quick checklist before submitting
- [ ] Run all cells top-to-bottom (`Kernel → Restart & Run All`) with no errors
- [ ] All figures visible at 100% zoom in the report PDF
- [ ] ARIMA order noted in the report (printed in Section 8)
- [ ] Model comparison table included in Results section
- [ ] AI declaration completed in Appendix A
- [ ] This notebook included as Appendix B